# EduArn Pandas — Reading CSV, JSON & Excel Files
## Complete Google Colab Teaching Notebook

This notebook teaches students how to load, inspect, clean, analyze and export data using:

- CSV
- JSON
- Excel (`.xlsx`)

We use the same **Employee Sales** business dataset throughout the lesson.

### Learning flow

**Create example files → Upload files in Colab → Read CSV → Read JSON → Read Excel → Inspect → Select → Filter → Analyze → Compare formats → Export results**


## 1. Setup

Pandas provides dedicated functions for common file formats:

- `pd.read_csv()` → CSV
- `pd.read_json()` → JSON
- `pd.read_excel()` → Excel

For Google Colab, files can be uploaded using `files.upload()`.

**Teaching tip:** Explain that Pandas converts external file data into a DataFrame. Once the data becomes a DataFrame, the same Pandas operations such as `head()`, filtering, sorting and `groupby()` can be used regardless of the original file format.


In [ ]:
import pandas as pd
import json
from pathlib import Path

print("Pandas version:", pd.__version__)


# 2. Create Example Files

To make this notebook completely runnable, we first create three example files:

1. `employee_sales.csv`
2. `employee_sales.json`
3. `employee_sales.xlsx`

Students can download these files and later practice uploading them into Google Colab.


In [ ]:
# Create a realistic business dataset
data = [
    {"Employee_ID": 101, "Employee": "Asha", "Department": "Sales", "City": "Hyderabad", "Experience": 3, "Sales": 120000, "Rating": 4.2},
    {"Employee_ID": 102, "Employee": "Rahul", "Department": "IT", "City": "Bengaluru", "Experience": 6, "Sales": 180000, "Rating": 4.7},
    {"Employee_ID": 103, "Employee": "Priya", "Department": "Sales", "City": "Hyderabad", "Experience": 4, "Sales": 150000, "Rating": 4.5},
    {"Employee_ID": 104, "Employee": "Arun", "Department": "HR", "City": "Chennai", "Experience": 8, "Sales": 90000, "Rating": 3.8},
    {"Employee_ID": 105, "Employee": "Neha", "Department": "IT", "City": "Bengaluru", "Experience": 5, "Sales": 210000, "Rating": 4.9},
    {"Employee_ID": 106, "Employee": "Vikram", "Department": "Sales", "City": "Pune", "Experience": 7, "Sales": 195000, "Rating": 4.4},
    {"Employee_ID": 107, "Employee": "Meena", "Department": "HR", "City": "Chennai", "Experience": 2, "Sales": 85000, "Rating": 3.9},
    {"Employee_ID": 108, "Employee": "Kiran", "Department": "IT", "City": "Hyderabad", "Experience": 9, "Sales": 230000, "Rating": 4.8}
]

df_example = pd.DataFrame(data)
display(df_example)


In [ ]:
# Save CSV, JSON and Excel examples
from pathlib import Path

example_dir = Path("/content/eduarn_pandas_file_examples")
example_dir.mkdir(exist_ok=True)

# CSV
csv_path = example_dir / "employee_sales.csv"
df_example.to_csv(csv_path, index=False)

# JSON
json_path = example_dir / "employee_sales.json"
df_example.to_json(json_path, orient="records", indent=4)

# Excel with two worksheets
department_summary = (
    df_example.groupby("Department")
    .agg(
        Employee_Count=("Employee_ID", "count"),
        Total_Sales=("Sales", "sum"),
        Average_Sales=("Sales", "mean"),
        Average_Rating=("Rating", "mean")
    )
    .round(2)
    .reset_index()
)

xlsx_path = example_dir / "employee_sales.xlsx"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_example.to_excel(writer, sheet_name="Employees", index=False)
    department_summary.to_excel(writer, sheet_name="Department_Summary", index=False)

print("Files created:")
print(csv_path)
print(json_path)
print(xlsx_path)


## 3. View the Example Files

In Google Colab, the following code displays the files created in the working directory.


In [ ]:
import os

for file in example_dir.iterdir():
    print(file.name, "->", file.stat().st_size, "bytes")


# 4. Reading a CSV File

## What is CSV?

CSV means **Comma-Separated Values**.

Example:

```text
Employee_ID,Employee,Department,City,Sales
101,Asha,Sales,Hyderabad,120000
102,Rahul,IT,Bengaluru,180000
```

CSV is one of the most common formats for exchanging tabular data.

### Main command

```python
pd.read_csv("file.csv")
```


In [ ]:
# Read CSV
csv_df = pd.read_csv(csv_path)

display(csv_df)


In [ ]:
# Inspect the CSV DataFrame
print("Shape:", csv_df.shape)
print("Columns:", csv_df.columns.tolist())

csv_df.head()


## CSV — Useful Options

You will frequently see:

```python
pd.read_csv("file.csv")
pd.read_csv("file.csv", sep=";")
pd.read_csv("file.csv", nrows=5)
pd.read_csv("file.csv", usecols=["Employee", "Sales"])
```

The most important lesson is:

> **Read first, inspect second, transform third.**


In [ ]:
# Read only selected columns
csv_selected = pd.read_csv(
    csv_path,
    usecols=["Employee", "Department", "Sales"]
)

display(csv_selected)


In [ ]:
# Read only the first 5 rows
csv_first5 = pd.read_csv(csv_path, nrows=5)
display(csv_first5)


# 5. Reading a JSON File

## What is JSON?

JSON means **JavaScript Object Notation**.

It is widely used in:

- REST APIs
- Web applications
- Cloud services
- Configuration
- Data exchange

Our example uses JSON records:

```json
[
    {
        "Employee_ID": 101,
        "Employee": "Asha",
        "Department": "Sales"
    }
]
```


In [ ]:
# Read JSON
json_df = pd.read_json(json_path)

display(json_df)


In [ ]:
# Inspect JSON data
print("Shape:", json_df.shape)
print("Columns:", json_df.columns.tolist())

json_df.info()


## JSON Orientation

JSON can have different structures.

Common Pandas options include:

```python
pd.read_json(file)
pd.read_json(file, orient="records")
```

For APIs, you may also receive nested JSON. In that case, `json.loads()` and `pd.json_normalize()` can be useful.


In [ ]:
# Demonstration of nested JSON
nested_json = {
    "employees": [
        {"id": 101, "name": "Asha", "skills": ["Python", "SQL"]},
        {"id": 102, "name": "Rahul", "skills": ["AWS", "Docker"]}
    ]
}

nested_df = pd.json_normalize(nested_json["employees"])

display(nested_df)


# 6. Reading an Excel File

Excel files can contain multiple worksheets.

Our example contains:

- `Employees`
- `Department_Summary`

### Main command

```python
pd.read_excel("file.xlsx")
```


In [ ]:
# Read the first Excel sheet
excel_df = pd.read_excel(xlsx_path)

display(excel_df)


In [ ]:
# Read a specific sheet
department_df = pd.read_excel(
    xlsx_path,
    sheet_name="Department_Summary"
)

display(department_df)


## Read All Excel Sheets

You can load all worksheets at once using:

```python
pd.read_excel("file.xlsx", sheet_name=None)
```

This returns a dictionary where:

- key = worksheet name
- value = DataFrame


In [ ]:
# Read every worksheet
all_sheets = pd.read_excel(xlsx_path, sheet_name=None)

print("Available sheets:", list(all_sheets.keys()))

for sheet_name, sheet_df in all_sheets.items():
    print("\n---", sheet_name, "---")
    display(sheet_df.head())


# 7. Compare CSV, JSON and Excel

A very important teaching point:

**Different file formats → same DataFrame concept.**

After loading the files, students can use the same commands:

```python
df.head()
df.shape
df.info()
df[df["Sales"] > 150000]
df.groupby("Department")["Sales"].sum()
```


In [ ]:
# Same analysis on all three formats

print("CSV total sales:", csv_df["Sales"].sum())
print("JSON total sales:", json_df["Sales"].sum())
print("Excel total sales:", excel_df["Sales"].sum())


In [ ]:
# Same filter on each DataFrame
print("CSV:")
display(csv_df[csv_df["Sales"] > 150000])

print("JSON:")
display(json_df[json_df["Sales"] > 150000])

print("Excel:")
display(excel_df[excel_df["Sales"] > 150000])


# 8. Practical Data Analysis After Reading a File

Once data is loaded, the normal workflow is:

### Step 1 — Inspect
`head()`, `shape`, `info()`

### Step 2 — Clean
Missing values, duplicates, data types

### Step 3 — Filter
Business conditions

### Step 4 — Transform
Create calculated columns

### Step 5 — Analyze
Statistics and GroupBy

### Step 6 — Export
CSV, Excel or JSON


In [ ]:
# Inspect
df = csv_df.copy()

print("Shape:", df.shape)
df.info()

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicates:", df.duplicated().sum())


In [ ]:
# Business filter
high_performers = df[
    (df["Sales"] >= 180000) &
    (df["Rating"] >= 4.5)
].copy()

display(high_performers)


In [ ]:
# Add calculated column
df["Bonus"] = df["Sales"] * 0.10

display(df[["Employee", "Sales", "Bonus"]])


In [ ]:
# GroupBy analysis
department_report = (
    df.groupby("Department")
      .agg(
          Employees=("Employee_ID", "count"),
          Total_Sales=("Sales", "sum"),
          Average_Sales=("Sales", "mean"),
          Average_Rating=("Rating", "mean")
      )
      .round(2)
      .sort_values("Total_Sales", ascending=False)
)

display(department_report)


# 9. Exporting Data

Pandas can also write DataFrames back to common file formats.

### CSV

```python
df.to_csv("output.csv", index=False)
```

### JSON

```python
df.to_json("output.json", orient="records", indent=4)
```

### Excel

```python
df.to_excel("output.xlsx", index=False)
```


In [ ]:
# Export cleaned/processed data

output_dir = Path("/content/eduarn_pandas_outputs")
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "employee_sales_processed.csv", index=False)
df.to_json(output_dir / "employee_sales_processed.json", orient="records", indent=4)
df.to_excel(output_dir / "employee_sales_processed.xlsx", index=False)

print("Export completed:")
for f in output_dir.iterdir():
    print(f.name)


# 10. Google Colab — Upload Your Own File

When teaching students, this is the most useful practical step.

Run:

```python
from google.colab import files
uploaded = files.upload()
```

Then identify the extension and use the appropriate Pandas reader.

### CSV

```python
df = pd.read_csv("your_file.csv")
```

### JSON

```python
df = pd.read_json("your_file.json")
```

### Excel

```python
df = pd.read_excel("your_file.xlsx")
```


In [ ]:
# Uncomment and run this cell in Google Colab when you want to upload your own file.

# from google.colab import files
# uploaded = files.upload()
#
# print("Uploaded files:")
# for filename in uploaded.keys():
#     print(filename)


# 11. Automatic File-Type Reader

This is a useful real-world pattern.

The function checks the extension and chooses the correct Pandas reader.


In [ ]:
def load_data_file(filename):
    """Load CSV, JSON or Excel file into a Pandas DataFrame."""

    extension = Path(filename).suffix.lower()

    if extension == ".csv":
        return pd.read_csv(filename)

    elif extension == ".json":
        return pd.read_json(filename)

    elif extension in [".xlsx", ".xls"]:
        return pd.read_excel(filename)

    else:
        raise ValueError(
            f"Unsupported file type: {extension}. "
            "Use CSV, JSON or Excel."
        )

# Test it
test_df = load_data_file(csv_path)
display(test_df.head())


# 12. Mini Project — Sales Data Pipeline

### Scenario

You receive an employee sales file from another team.

Your job is to:

1. Load the file.
2. Inspect the data.
3. Find missing values.
4. Remove duplicates.
5. Find employees with sales above ₹150,000.
6. Calculate a 10% bonus.
7. Find total sales by department.
8. Export the final report to Excel.


In [ ]:
# Complete mini project

project_df = pd.read_csv(csv_path)

# 1. Inspect
print("Initial shape:", project_df.shape)

# 2. Missing values
print("\nMissing values:")
print(project_df.isna().sum())

# 3. Remove duplicates
project_df = project_df.drop_duplicates()

# 4. Filter high sales
high_sales = project_df[project_df["Sales"] > 150000].copy()

# 5. Calculate bonus
high_sales["Bonus"] = high_sales["Sales"] * 0.10

# 6. Department report
report = (
    project_df.groupby("Department")
    .agg(
        Employees=("Employee_ID", "count"),
        Total_Sales=("Sales", "sum"),
        Average_Sales=("Sales", "mean")
    )
    .round(2)
    .reset_index()
)

print("\nHigh-sales employees:")
display(high_sales)

print("\nDepartment report:")
display(report)


In [ ]:
# Export mini-project result to Excel

final_report_path = output_dir / "EduArn_Sales_Final_Report.xlsx"

with pd.ExcelWriter(final_report_path, engine="openpyxl") as writer:
    project_df.to_excel(writer, sheet_name="Clean_Data", index=False)
    high_sales.to_excel(writer, sheet_name="High_Sales", index=False)
    report.to_excel(writer, sheet_name="Department_Report", index=False)

print("Final report created:", final_report_path)


# 13. Student Practice Exercises

### Beginner

1. Read `employee_sales.csv`.
2. Display the first 3 rows.
3. Display only `Employee` and `Sales`.
4. Find the shape of the DataFrame.

### Intermediate

5. Read the JSON file.
6. Read only the `Employees` sheet from Excel.
7. Find employees whose Sales are greater than ₹180,000.
8. Find average Sales by Department.
9. Sort employees from highest to lowest Sales.

### Advanced

10. Create a `Performance_Level` column:
   - Rating >= 4.5 → Excellent
   - Rating >= 4.0 → Good
   - Otherwise → Needs Improvement

11. Create a department dashboard.
12. Export the dashboard to Excel with separate sheets.
13. Modify the automatic file reader so it also supports `.txt` files containing comma-separated data.


# Final Cheat Sheet

| Requirement | Pandas |
|---|---|
| Read CSV | `pd.read_csv()` |
| Read JSON | `pd.read_json()` |
| Read Excel | `pd.read_excel()` |
| Read Excel sheet | `pd.read_excel(sheet_name="...")` |
| Read all Excel sheets | `pd.read_excel(sheet_name=None)` |
| Inspect | `df.head()` |
| Structure | `df.info()` |
| Dimensions | `df.shape` |
| Filter | `df[df["Sales"] > 150000]` |
| Group | `df.groupby("Department")` |
| Export CSV | `df.to_csv()` |
| Export JSON | `df.to_json()` |
| Export Excel | `df.to_excel()` |

## Remember

**File → DataFrame → Inspect → Clean → Transform → Analyze → Export**

Once data is converted into a Pandas DataFrame, the majority of your Pandas skills work the same way regardless of whether the original data came from CSV, JSON or Excel.
